In [ ]:

import os
import time
from pathlib import Path

from downloader.genome_downloader import GgetEnsemblGenomeDownloader
from genome.builder import GenomeBuilder

# Define parameters for the test
ensembl_release = 114 
assembly_id = "GRCh38"
species = "homo_sapiens"

# Track total execution time
total_start_time = time.time()

# 1. Download the files
print(f"Starting download for {species} (release {ensembl_release})...")
download_start_time = time.time()

downloader = GgetEnsemblGenomeDownloader(assembly_id=assembly_id, ensembl_release=ensembl_release, species=species)
downloaded_files = downloader.download()

download_end_time = time.time()
download_duration = download_end_time - download_start_time
print(f"Download completed in: {download_duration:.2f} seconds")

dna_path = downloaded_files['dna']
cdna_path = downloaded_files['cdna']
gtf_path = downloaded_files['annotation']

# 2. Build the Genome object from the downloaded files
print("\nBuilding Genome object...")
build_start_time = time.time()

builder = GenomeBuilder(id=assembly_id, species=species, name=f"{species} Genome (release {ensembl_release})")

# Time each step separately
print("  Step 1: Loading DNA FASTA...")
dna_start_time = time.time()
builder = builder.with_dna_fasta(dna_path)
dna_end_time = time.time()
dna_duration = dna_end_time - dna_start_time
print(f"    DNA loading completed in: {dna_duration:.2f} seconds")


Starting download for homo_sapiens (release 114)...


INFO:GenomeBuilder:Loading DNA sequences from data/GRCh38/114/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz...
INFO:GenomeBuilder:Using existing extracted DNA FASTA file: data/GRCh38/114/Homo_sapiens.GRCh38.dna.primary_assembly.fa


Download completed in: 1.49 seconds

Building Genome object...
  Step 1: Loading DNA FASTA...


In [ ]:

print("  Step 2: Loading cDNA FASTA...")
cdna_start_time = time.time()
builder = builder.with_cdna_fasta(cdna_path)
cdna_end_time = time.time()
cdna_duration = cdna_end_time - cdna_start_time
print(f"    cDNA loading completed in: {cdna_duration:.2f} seconds")


INFO:GenomeBuilder:Loading cDNA sequences from data/GRCh38/114/Homo_sapiens.GRCh38.cdna.all.fa.gz...
INFO:GenomeBuilder:Reading gzipped cDNA FASTA file: data/GRCh38/114/Homo_sapiens.GRCh38.cdna.all.fa.gz


  Step 2: Loading cDNA FASTA...


INFO:GenomeBuilder:Loaded 207175 cDNA sequences.


    cDNA loading completed in: 4.78 seconds


In [ ]:

print("  Step 3: Loading GTF annotation...")
gtf_start_time = time.time()
builder = builder.with_gtf_file(gtf_path)
gtf_end_time = time.time()
gtf_duration = gtf_end_time - gtf_start_time
print(f"    GTF loading completed in: {gtf_duration:.2f} seconds")


INFO:GenomeBuilder:Processing annotations from data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.gz...
INFO:GenomeBuilder:Loading existing gffutils database: data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.db
INFO:root:GTF database created at: data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.db


  Step 3: Loading GTF annotation...


INFO:GenomeBuilder:Created genes in 7.07 seconds
INFO:GenomeBuilder:Created transcripts in 34.25 seconds
INFO:GenomeBuilder:Created exons in 128.05 seconds
INFO:GenomeBuilder:Successfully parsed and linked 78894 genes, 387954 transcripts.


    GTF loading completed in: 169.43 seconds


In [ ]:

print("  Step 4: Building final genome object...")
final_build_start_time = time.time()
genome = builder.build()
final_build_end_time = time.time()
final_build_duration = final_build_end_time - final_build_start_time
print(f"    Final build completed in: {final_build_duration:.2f} seconds")

INFO:GenomeBuilder:Indexing genome for fast lookups...


  Step 4: Building final genome object...


INFO:GenomeBuilder:Genome construction complete.
INFO:GenomeBuilder:Offloading builder memory...
INFO:GenomeBuilder:Memory offload complete.


    Final build completed in: 6.81 seconds


In [10]:
import sys
from collections.abc import Mapping, Container

def get_deep_size_skip_parent(obj, seen=None):
    """
    Recursively calculates the total memory usage of an object and all its contents,
    but skips _parent attributes to avoid circular references.
    """
    if seen is None:
        seen = set()
    
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    
    # Mark this object as seen
    seen.add(obj_id)
    
    # Get the size of the object itself
    size = sys.getsizeof(obj)
    
    # If it's a string, number, or other simple type, we're done
    if isinstance(obj, (str, bytes, bytearray, int, float, complex, bool, type(None))):
        return size
    
    # For containers, recursively calculate the size of their contents
    if isinstance(obj, dict):
        for k, v in obj.items():
            # Skip _parent keys
            if k != '_parent':
                size += get_deep_size_skip_parent(k, seen) + get_deep_size_skip_parent(v, seen)
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(get_deep_size_skip_parent(item, seen) for item in obj)
    elif hasattr(obj, '__dict__'):
        # For custom objects, check their __dict__ but skip _parent
        for attr_name, attr_value in obj.__dict__.items():
            if attr_name != '_parent':
                size += get_deep_size_skip_parent(attr_value, seen)
    elif hasattr(obj, '__slots__'):
        # For objects with __slots__, check each slot but skip _parent
        for slot in obj.__slots__:
            if slot != '_parent' and hasattr(obj, slot):
                size += get_deep_size_skip_parent(getattr(obj, slot), seen)
    
    return size

def format_bytes(bytes_size):
    """Convert bytes to human readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PB"

# # Calculate the deep size of the genome object (skipping _parent references)
# genome_size = get_deep_size_skip_parent(genome)
# print(f"Total memory usage of genome object (skipping _parent): {format_bytes(genome_size)}")
# print(f"Raw bytes: {genome_size:,}")



In [11]:
from collections import defaultdict

def get_memory_breakdown_by_type(obj, seen=None, type_stats=None):
    """
    Recursively calculates memory usage broken down by object type,
    skipping _parent attributes and _seq_index in chromosome objects.
    """
    if seen is None:
        seen = set()
    if type_stats is None:
        type_stats = defaultdict(lambda: {'count': 0, 'total_size': 0})
    
    obj_id = id(obj)
    if obj_id in seen:
        return type_stats
    
    # Mark this object as seen
    seen.add(obj_id)
    
    # Get the size of the object itself
    size = sys.getsizeof(obj)
    obj_type = type(obj).__name__
    
    # Track this object
    type_stats[obj_type]['count'] += 1
    type_stats[obj_type]['total_size'] += size
    
    # If it's a string, number, or other simple type, we're done
    if isinstance(obj, (str, bytes, bytearray, int, float, complex, bool, type(None))):
        return type_stats
    
    # For containers, recursively analyze their contents
    if isinstance(obj, dict):
        for k, v in obj.items():
            # Skip _parent keys and _seq_index
            if k not in ('_parent', '_seq_index'):
                get_memory_breakdown_by_type(k, seen, type_stats)
                get_memory_breakdown_by_type(v, seen, type_stats)
    elif isinstance(obj, (list, tuple, set, frozenset)):
        for item in obj:
            get_memory_breakdown_by_type(item, seen, type_stats)
    elif hasattr(obj, '__dict__'):
        # For custom objects, check their __dict__ but skip _parent and _seq_index
        for attr_name, attr_value in obj.__dict__.items():
            if attr_name not in ('_parent', '_seq_index'):
                get_memory_breakdown_by_type(attr_value, seen, type_stats)
    elif hasattr(obj, '__slots__'):
        # For objects with __slots__, check each slot but skip _parent and _seq_index
        for slot in obj.__slots__:
            if slot not in ('_parent', '_seq_index') and hasattr(obj, slot):
                get_memory_breakdown_by_type(getattr(obj, slot), seen, type_stats)
    
    return type_stats

def print_memory_breakdown(type_stats):
    """Print memory breakdown sorted by total size."""
    print("\n=== Memory Breakdown by Object Type ===")
    print(f"{'Type':<20} {'Count':<10} {'Total Size':<15} {'Avg Size':<15}")
    print("-" * 65)
    
    # Sort by total size (descending)
    sorted_types = sorted(type_stats.items(), key=lambda x: x[1]['total_size'], reverse=True)
    
    total_objects = sum(stats['count'] for stats in type_stats.values())
    total_memory = sum(stats['total_size'] for stats in type_stats.values())
    
    for obj_type, stats in sorted_types:
        count = stats['count']
        total_size = stats['total_size']
        avg_size = total_size / count if count > 0 else 0
        percentage = (total_size / total_memory) * 100 if total_memory > 0 else 0
        
        print(f"{obj_type:<20} {count:<10,} {format_bytes(total_size):<15} {format_bytes(avg_size):<15} ({percentage:.1f}%)")
    
    print("-" * 65)
    print(f"{'TOTAL':<20} {total_objects:<10,} {format_bytes(total_memory):<15}")
    
    return sorted_types, total_memory

# Analyze memory breakdown by type
print("Analyzing memory usage by object type...")
type_stats = get_memory_breakdown_by_type(genome)
sorted_types, total_memory = print_memory_breakdown(type_stats)

print(f"\nTotal analyzed memory: {format_bytes(total_memory)}")
print(f"Raw bytes: {total_memory:,}")


Analyzing memory usage by object type...

=== Memory Breakdown by Object Type ===
Type                 Count      Total Size      Avg Size       
-----------------------------------------------------------------
str                  55,621,533 3.16 GB         60.92 B         (42.7%)
list                 32,214,617 2.60 GB         86.50 B         (35.2%)
dict                 2,632,143  1.20 GB         488.17 B        (16.2%)
Locus                2,632,138  140.57 MB       56.00 B         (1.9%)
int                  5,264,078  140.57 MB       28.00 B         (1.9%)
Exon                 2,165,096  115.63 MB       56.00 B         (1.5%)
Seq                  387,954    23.68 MB        64.00 B         (0.3%)
Transcript           387,954    20.72 MB        56.00 B         (0.3%)
Gene                 78,894     4.21 MB         56.00 B         (0.1%)
Chromosome           194        10.61 KB        56.00 B         (0.0%)
Genome               1          56.00 B         56.00 B         (0.0%)
bool

In [12]:
def analyze_genome_components(genome):
    """
    Detailed analysis of genome components and their memory usage.
    """
    print("\n=== Detailed Genome Component Analysis ===")
    
    # Analyze chromosomes
    chromosome_count = len(genome.chromosomes)
    chromosome_sizes = []
    total_chromosome_memory = 0
    
    print(f"\nChromosomes: {chromosome_count}")
    for chrom_id, chrom in genome.chromosomes.items():
        chrom_size = get_deep_size_skip_parent(chrom)
        chromosome_sizes.append(chrom_size)
        total_chromosome_memory += chrom_size
    
    if chromosome_sizes:
        avg_chrom_size = sum(chromosome_sizes) / len(chromosome_sizes)
        max_chrom_size = max(chromosome_sizes)
        min_chrom_size = min(chromosome_sizes)
        print(f"  Total memory: {format_bytes(total_chromosome_memory)}")
        print(f"  Average per chromosome: {format_bytes(avg_chrom_size)}")
        print(f"  Largest chromosome: {format_bytes(max_chrom_size)}")
        print(f"  Smallest chromosome: {format_bytes(min_chrom_size)}")
    
    # Analyze genes
    genes = list(genome.genes_iter())
    gene_count = len(genes)
    gene_sizes = []
    total_gene_memory = 0
    
    print(f"\nGenes: {gene_count}")
    for gene in genes[:1000]:  # Sample first 1000 genes to avoid too much computation
        gene_size = get_deep_size_skip_parent(gene)
        gene_sizes.append(gene_size)
        total_gene_memory += gene_size
    
    if gene_sizes:
        avg_gene_size = sum(gene_sizes) / len(gene_sizes)
        max_gene_size = max(gene_sizes)
        min_gene_size = min(gene_sizes)
        estimated_total_gene_memory = avg_gene_size * gene_count
        print(f"  Estimated total memory: {format_bytes(estimated_total_gene_memory)} (based on {len(gene_sizes)} samples)")
        print(f"  Average per gene: {format_bytes(avg_gene_size)}")
        print(f"  Largest gene (sampled): {format_bytes(max_gene_size)}")
        print(f"  Smallest gene (sampled): {format_bytes(min_gene_size)}")
    
    # Analyze transcripts
    transcripts = list(genome.transcripts_iter())
    transcript_count = len(transcripts)
    transcript_sizes = []
    total_transcript_memory = 0
    
    print(f"\nTranscripts: {transcript_count}")
    for transcript in transcripts[:1000]:  # Sample first 1000 transcripts
        transcript_size = get_deep_size_skip_parent(transcript)
        transcript_sizes.append(transcript_size)
        total_transcript_memory += transcript_size
    
    if transcript_sizes:
        avg_transcript_size = sum(transcript_sizes) / len(transcript_sizes)
        max_transcript_size = max(transcript_sizes)
        min_transcript_size = min(transcript_sizes)
        estimated_total_transcript_memory = avg_transcript_size * transcript_count
        print(f"  Estimated total memory: {format_bytes(estimated_total_transcript_memory)} (based on {len(transcript_sizes)} samples)")
        print(f"  Average per transcript: {format_bytes(avg_transcript_size)}")
        print(f"  Largest transcript (sampled): {format_bytes(max_transcript_size)}")
        print(f"  Smallest transcript (sampled): {format_bytes(min_transcript_size)}")
    
    # Analyze exons
    exons = list(genome.exons_iter())
    exon_count = len(exons)
    exon_sizes = []
    total_exon_memory = 0
    
    print(f"\nExons: {exon_count}")
    for exon in exons[:1000]:  # Sample first 1000 exons
        exon_size = get_deep_size_skip_parent(exon)
        exon_sizes.append(exon_size)
        total_exon_memory += exon_size
    
    if exon_sizes:
        avg_exon_size = sum(exon_sizes) / len(exon_sizes)
        max_exon_size = max(exon_sizes)
        min_exon_size = min(exon_sizes)
        estimated_total_exon_memory = avg_exon_size * exon_count
        print(f"  Estimated total memory: {format_bytes(estimated_total_exon_memory)} (based on {len(exon_sizes)} samples)")
        print(f"  Average per exon: {format_bytes(avg_exon_size)}")
        print(f"  Largest exon (sampled): {format_bytes(max_exon_size)}")
        print(f"  Smallest exon (sampled): {format_bytes(min_exon_size)}")
    
    # Analyze lookup dictionaries
    print(f"\nLookup Dictionaries:")
    genes_dict_size = get_deep_size_skip_parent(genome._genes_by_id)
    transcripts_dict_size = get_deep_size_skip_parent(genome._transcripts_by_id)
    exons_dict_size = get_deep_size_skip_parent(genome._exons_by_id)
    
    print(f"  Genes lookup dict: {format_bytes(genes_dict_size)} ({len(genome._genes_by_id)} entries)")
    print(f"  Transcripts lookup dict: {format_bytes(transcripts_dict_size)} ({len(genome._transcripts_by_id)} entries)")
    print(f"  Exons lookup dict: {format_bytes(exons_dict_size)} ({len(genome._exons_by_id)} entries)")
    
    return {
        'chromosome_memory': total_chromosome_memory,
        'estimated_gene_memory': estimated_total_gene_memory if gene_sizes else 0,
        'estimated_transcript_memory': estimated_total_transcript_memory if transcript_sizes else 0,
        'estimated_exon_memory': estimated_total_exon_memory if exon_sizes else 0,
        'genes_dict_memory': genes_dict_size,
        'transcripts_dict_memory': transcripts_dict_size,
        'exons_dict_memory': exons_dict_size
    }

# Run the detailed analysis
component_breakdown = analyze_genome_components(genome)



=== Detailed Genome Component Analysis ===

Chromosomes: 194
  Total memory: 7.50 GB
  Average per chromosome: 39.61 MB
  Largest chromosome: 701.47 MB
  Smallest chromosome: 1.03 MB

Genes: 78894
  Estimated total memory: 5.23 GB (based on 1000 samples)
  Average per gene: 69.49 KB
  Largest gene (sampled): 1.75 MB
  Smallest gene (sampled): 6.39 KB

Transcripts: 387954
  Estimated total memory: 10.42 GB (based on 1000 samples)
  Average per transcript: 28.16 KB
  Largest transcript (sampled): 267.25 KB
  Smallest transcript (sampled): 5.22 KB

Exons: 2165096
  Estimated total memory: 7.50 GB (based on 1000 samples)
  Average per exon: 3.63 KB
  Largest exon (sampled): 4.11 KB
  Smallest exon (sampled): 2.79 KB

Lookup Dictionaries:
  Genes lookup dict: 7.31 GB (78894 entries)
  Transcripts lookup dict: 7.23 GB (387954 entries)
  Exons lookup dict: 6.33 GB (2165096 entries)


In [13]:
def analyze_heavy_attributes(genome):
    """
    Analyze specific attributes that are likely consuming the most memory.
    """
    print("\n=== Heavy Attribute Analysis ===")
    
    # Check SeqIO.index objects in chromosomes
    seq_index_memory = 0
    seq_index_count = 0
    
    print("\nSeqIO.index objects in chromosomes:")
    for chrom_id, chrom in list(genome.chromosomes.items())[:5]:  # Check first 5 chromosomes
        if hasattr(chrom, '_seq_index') and chrom._seq_index is not None:
            index_size = sys.getsizeof(chrom._seq_index)
            seq_index_memory += index_size
            seq_index_count += 1
            print(f"  {chrom_id}: {format_bytes(index_size)}")
    
    if seq_index_count > 0:
        estimated_total_seq_index = (seq_index_memory / seq_index_count) * len(genome.chromosomes)
        print(f"  Estimated total SeqIO.index memory: {format_bytes(estimated_total_seq_index)}")
    
    # Check Bio.Seq objects in transcripts
    seq_objects_memory = 0
    seq_objects_count = 0
    
    print("\nBio.Seq objects in transcripts (sample):")
    transcripts = list(genome.transcripts_iter())[:100]  # Sample 100 transcripts
    for transcript in transcripts:
        if hasattr(transcript, 'sequence') and transcript.sequence is not None:
            seq_size = sys.getsizeof(transcript.sequence)
            seq_objects_memory += seq_size
            seq_objects_count += 1
    
    if seq_objects_count > 0:
        avg_seq_size = seq_objects_memory / seq_objects_count
        estimated_total_seq_memory = avg_seq_size * len(list(genome.transcripts_iter()))
        print(f"  Average Bio.Seq size: {format_bytes(avg_seq_size)}")
        print(f"  Estimated total Bio.Seq memory: {format_bytes(estimated_total_seq_memory)} (based on {seq_objects_count} samples)")
    
    # Check string attributes in annotations
    print("\nString attributes in genome features:")
    
    # Sample some objects and check their string attributes
    sample_gene = next(genome.genes_iter())
    sample_transcript = next(genome.transcripts_iter())
    sample_exon = next(genome.exons_iter())
    
    def analyze_string_attrs(obj, obj_type):
        total_str_size = 0
        str_attrs = []
        if hasattr(obj, '__dict__'):
            for attr_name, attr_value in obj.__dict__.items():
                if isinstance(attr_value, str):
                    str_size = sys.getsizeof(attr_value)
                    total_str_size += str_size
                    str_attrs.append((attr_name, str_size, len(attr_value)))
        
        if hasattr(obj, '_attributes') and isinstance(obj._attributes, dict):
            for attr_name, attr_value in obj._attributes.items():
                if isinstance(attr_value, (str, list)):
                    if isinstance(attr_value, list) and attr_value and isinstance(attr_value[0], str):
                        # Handle list of strings (common in GTF attributes)
                        for s in attr_value:
                            str_size = sys.getsizeof(s)
                            total_str_size += str_size
                            str_attrs.append((f"{attr_name}[item]", str_size, len(s)))
                    elif isinstance(attr_value, str):
                        str_size = sys.getsizeof(attr_value)
                        total_str_size += str_size
                        str_attrs.append((attr_name, str_size, len(attr_value)))
        
        print(f"  {obj_type} string attributes total: {format_bytes(total_str_size)}")
        for attr_name, size, length in sorted(str_attrs, key=lambda x: x[1], reverse=True)[:3]:
            print(f"    {attr_name}: {format_bytes(size)} ({length} chars)")
        return total_str_size
    
    gene_str_size = analyze_string_attrs(sample_gene, "Gene")
    transcript_str_size = analyze_string_attrs(sample_transcript, "Transcript")
    exon_str_size = analyze_string_attrs(sample_exon, "Exon")
    
    # Estimate total string memory
    total_genes = len(list(genome.genes_iter()))
    total_transcripts = len(list(genome.transcripts_iter()))
    total_exons = len(list(genome.exons_iter()))
    
    estimated_gene_strings = gene_str_size * total_genes
    estimated_transcript_strings = transcript_str_size * total_transcripts
    estimated_exon_strings = exon_str_size * total_exons
    
    print(f"\nEstimated total string memory:")
    print(f"  Genes: {format_bytes(estimated_gene_strings)}")
    print(f"  Transcripts: {format_bytes(estimated_transcript_strings)}")
    print(f"  Exons: {format_bytes(estimated_exon_strings)}")
    print(f"  Total strings: {format_bytes(estimated_gene_strings + estimated_transcript_strings + estimated_exon_strings)}")

# Run the heavy attribute analysis
analyze_heavy_attributes(genome)



=== Heavy Attribute Analysis ===

SeqIO.index objects in chromosomes:
  1: 56.00 B
  10: 56.00 B
  11: 56.00 B
  12: 56.00 B
  13: 56.00 B
  Estimated total SeqIO.index memory: 10.61 KB

Bio.Seq objects in transcripts (sample):
  Average Bio.Seq size: 64.00 B
  Estimated total Bio.Seq memory: 23.68 MB (based on 100 samples)

String attributes in genome features:
  Gene string attributes total: 296.00 B
    id: 64.00 B (15 chars)
    gene_source[item]: 63.00 B (14 chars)
    gene_biotype[item]: 63.00 B (14 chars)
  Transcript string attributes total: 699.00 B
    tag[item]: 64.00 B (15 chars)
    id: 64.00 B (15 chars)
    gene_source[item]: 63.00 B (14 chars)
  Exon string attributes total: 854.00 B
    exon_id[item]: 64.00 B (15 chars)
    tag[item]: 64.00 B (15 chars)
    gene_source[item]: 63.00 B (14 chars)

Estimated total string memory:
  Genes: 22.27 MB
  Transcripts: 258.62 MB
  Exons: 1.72 GB
  Total strings: 2.00 GB


In [7]:
import sys
sys.getsizeof(genome.exons)
print(len(genome.exons))

2165096


In [6]:
sys.getsizeof(genome.transcripts)

3292664